# 🚀 Day 3: Multi-Qubit Systems & Entanglement

## 14-Day Quantum DevRel Bootcamp

**Goal:** Understand the resource that makes quantum computing exponentially powerful — **entanglement**.

**Today's Deliverables:**
1. ✅ Build all 4 Bell state circuits
2. ✅ Verify entanglement with partial trace
3. ✅ Visualize why entangled qubits lose individual identity
4. ✅ GHZ vs W states — two types of multi-qubit entanglement
5. ✅ No-cloning theorem demonstration
6. ✅ Bell state measurement correlations

**Key insight:** Yesterday you mastered single-qubit rotations.  
Today you'll see that **two qubits together can do things no pair of single qubits can**.

---

## Section 1: Multi-Qubit State Spaces (Tensor Products)

In [ ]:
# Core imports
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, Operator

print("✅ All imports loaded!")
print()

# ── Tensor product structure ──
print("📐 Multi-Qubit State Spaces")
print("=" * 55)
print()
print("1 qubit:  state in ℂ² (2 amplitudes)")
print("2 qubits: state in ℂ² ⊗ ℂ² = ℂ⁴ (4 amplitudes)")
print("3 qubits: state in ℂ⁸ (8 amplitudes)")
print("n qubits: state in ℂ^(2ⁿ) amplitudes")
print()

# Build the 4 computational basis states for 2 qubits
print("Computational basis states:")
print("-" * 55)
basis_circuits = {
    '|00⟩': lambda: QuantumCircuit(2),
    '|01⟩': lambda: (qc := QuantumCircuit(2), qc.x(0))[0],  # qubit 0 = rightmost
    '|10⟩': lambda: (qc := QuantumCircuit(2), qc.x(1))[0],
    '|11⟩': lambda: (qc := QuantumCircuit(2), qc.x(0), qc.x(1))[0],
}

for name, make_qc in basis_circuits.items():
    sv = Statevector.from_instruction(make_qc())
    print(f"  {name} = {np.array(sv).real}")

print()

# Manual tensor product with np.kron
print("Manual tensor product: |+⟩ ⊗ |0⟩")
ket_plus = np.array([1, 1]) / np.sqrt(2)
ket_0 = np.array([1, 0])
product = np.kron(ket_plus, ket_0)
print(f"  np.kron(|+⟩, |0⟩) = {product.round(4)}")

# Verify with Qiskit
qc = QuantumCircuit(2)
qc.h(1)  # qubit 1 → |+⟩ (Qiskit: qubit 1 is leftmost in ket notation)
sv_qiskit = np.array(Statevector.from_instruction(qc))
print(f"  Qiskit |+0⟩     = {sv_qiskit.real.round(4)}")
print(f"  Match: {np.allclose(product, sv_qiskit)} ✅")
print()
print("💡 Two qubits → 4 amplitudes. But NOT all 4-amplitude states")
print("   can be written as |a⟩⊗|b⟩. Those that can't are ENTANGLED.")

---
## Section 2: The CNOT Gate

The **CNOT (Controlled-NOT)** gate is the fundamental two-qubit entangling gate.

$$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

**Action:** Flip the target qubit if the control qubit is $\vert 1\rangle$:

| Input | Output | Description |
| :--- | :--- | :--- |
| $\vert 00\rangle$ | $\vert 00\rangle$ | Control=0, no flip |
| $\vert 01\rangle$ | $\vert 01\rangle$ | Control=0, no flip |
| $\vert 10\rangle$ | $\vert 11\rangle$ | Control=1, target flipped |
| $\vert 11\rangle$ | $\vert 10\rangle$ | Control=1, target flipped |

⚠️ **CNOT alone doesn't create entanglement** from basis states.  
You need **superposition first** (e.g., H on control), THEN CNOT.

In [ ]:
print("🔗 The CNOT Gate")
print("=" * 55)

# CNOT matrix from Qiskit
qc_cnot = QuantumCircuit(2)
qc_cnot.cx(1, 0)  # control=1, target=0
cnot_matrix = Operator(qc_cnot).data
print("CNOT matrix:")
print(cnot_matrix.real.astype(int))
print()

# CNOT truth table
print("CNOT truth table (control=qubit 1, target=qubit 0):")
print("-" * 55)
for input_state in ['00', '01', '10', '11']:
    qc = QuantumCircuit(2)
    if input_state[0] == '1': qc.x(1)  # leftmost = qubit 1
    if input_state[1] == '1': qc.x(0)  # rightmost = qubit 0
    qc.cx(1, 0)  # CNOT: control=1, target=0
    sv = Statevector.from_instruction(qc)
    output = sv.probabilities_dict()
    result = max(output, key=output.get)
    print(f"  |{input_state}⟩ → |{result}⟩")

print()
print("CNOT circuit diagram:")
qc_demo = QuantumCircuit(2)
qc_demo.cx(1, 0)
print(qc_demo.draw())
print()

# Show: CNOT on basis states → no entanglement
print("Does CNOT on |10⟩ create entanglement?")
qc_basis = QuantumCircuit(2)
qc_basis.x(1)      # |10⟩
qc_basis.cx(1, 0)   # → |11⟩
sv = Statevector.from_instruction(qc_basis)
print(f"  CNOT|10⟩ = {np.array(sv).real}  (still a basis state — NOT entangled)")
print()

# Show: H + CNOT creates entanglement
print("Does H + CNOT on |00⟩ create entanglement?")
qc_ent = QuantumCircuit(2)
qc_ent.h(1)         # |00⟩ → |+0⟩ = (|00⟩ + |10⟩)/√2
qc_ent.cx(1, 0)     # → (|00⟩ + |11⟩)/√2 = |Φ+⟩
sv = Statevector.from_instruction(qc_ent)
print(f"  H⊗I then CNOT|00⟩ = {np.array(sv).real.round(4)}  → ENTANGLED! ✅")
print()
print("💡 Superposition + CNOT = Entanglement")
print("   H creates superposition, CNOT correlates the qubits.")

---
## Section 3: Bell States — The Four Maximally Entangled States

The Bell states are the **simplest entangled states** and form an orthonormal basis for 2 qubits:

$$\vert\Phi^+\rangle = \frac{\vert 00\rangle + \vert 11\rangle}{\sqrt{2}} \qquad \vert\Phi^-\rangle = \frac{\vert 00\rangle - \vert 11\rangle}{\sqrt{2}}$$

$$\vert\Psi^+\rangle = \frac{\vert 01\rangle + \vert 10\rangle}{\sqrt{2}} \qquad \vert\Psi^-\rangle = \frac{\vert 01\rangle - \vert 10\rangle}{\sqrt{2}}$$

### The Pattern
All 4 are built from `|00⟩` → optional X gates → **H** → **CNOT**:

| Bell State | Start From | Then Apply |
| :--- | :--- | :--- |
| $\vert\Phi^+\rangle$ | $\vert 00\rangle$ | H on q1, CNOT(1→0) |
| $\vert\Phi^-\rangle$ | $\vert 10\rangle$ | H on q1, CNOT(1→0) |
| $\vert\Psi^+\rangle$ | $\vert 01\rangle$ | H on q1, CNOT(1→0) |
| $\vert\Psi^-\rangle$ | $\vert 11\rangle$ | H on q1, CNOT(1→0) |

In [ ]:
print("🔔 The Four Bell States")
print("=" * 55)

def make_bell(name, x_qubits=None):
    """Create a Bell state circuit."""
    qc = QuantumCircuit(2)
    if x_qubits:
        for q in x_qubits:
            qc.x(q)
    qc.h(1)
    qc.cx(1, 0)
    return qc

bell_states = {
    '|Φ+⟩': make_bell('Φ+'),
    '|Φ-⟩': make_bell('Φ-', [1]),
    '|Ψ+⟩': make_bell('Ψ+', [0]),
    '|Ψ-⟩': make_bell('Ψ-', [0, 1]),
}

for name, qc in bell_states.items():
    sv = Statevector.from_instruction(qc)
    arr = np.array(sv)
    print(f"\n{name}:")
    print(f"  Circuit: {qc.draw(output='text', fold=-1)}")
    print(f"  State:   [{arr[0]:+.4f}, {arr[1]:+.4f}, {arr[2]:+.4f}, {arr[3]:+.4f}]")
    # Verify entanglement
    dm = DensityMatrix(sv)
    reduced = partial_trace(dm, [0])
    purity = np.real(np.trace(reduced.data @ reduced.data))
    print(f"  Purity of qubit 1: {purity:.4f} (0.5 = maximally entangled) ✅")

---
## Section 4: Visualizing Entanglement on the Bloch Sphere

**Key insight:** An entangled qubit has **no individual Bloch vector**.

- **Separable state** $\vert +0\rangle$: Each qubit has a definite Bloch vector  
  → Qubit A points to +X, Qubit B points to +Z
  
- **Entangled state** $\vert\Phi^+\rangle$: Individual qubits are **maximally mixed**  
  → Both Bloch vectors collapse to the **origin** (center of sphere)

The information is in the **correlations**, not in individual qubits!

In [ ]:
# ── Bloch sphere helpers (from Day 2) ──
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def dm_to_bloch(dm) -> tuple:
    """Convert a single-qubit density matrix to Bloch coordinates."""
    rho = dm.data if hasattr(dm, 'data') else dm
    x = np.real(np.trace(rho @ PAULI_X))
    y = np.real(np.trace(rho @ PAULI_Y))
    z = np.real(np.trace(rho @ PAULI_Z))
    return float(x), float(y), float(z)


def make_sphere_wireframe():
    """Create plotly traces for a translucent Bloch sphere."""
    traces = []
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(len(u)), np.cos(v))
    traces.append(go.Surface(
        x=xs, y=ys, z=zs, opacity=0.1,
        colorscale=[[0, 'lightgray'], [1, 'lightgray']],
        showscale=False, hoverinfo='skip'
    ))
    for coords, color, label in [
        ([1.4,0,0], 'red', 'X'), ([0,1.4,0], 'green', 'Y'), ([0,0,1.4], 'blue', 'Z|0⟩')
    ]:
        traces.append(go.Scatter3d(
            x=[0, coords[0]], y=[0, coords[1]], z=[0, coords[2]],
            mode='lines+text', text=['', label], textposition='top center',
            line=dict(color=color, width=3), showlegend=False, hoverinfo='skip'
        ))
    return traces

print("✅ Bloch sphere helpers defined.")

In [ ]:
# ── Compare: Separable vs Entangled ──
print("👁️ Visualizing: Separable vs Entangled Bloch Vectors")
print("=" * 55)

# Separable state: |+0⟩
qc_sep = QuantumCircuit(2)
qc_sep.h(1)
dm_sep = DensityMatrix(Statevector.from_instruction(qc_sep))
rho_A_sep = partial_trace(dm_sep, [0])  # qubit 1 (leftmost)
rho_B_sep = partial_trace(dm_sep, [1])  # qubit 0 (rightmost)
bloch_A_sep = dm_to_bloch(rho_A_sep)
bloch_B_sep = dm_to_bloch(rho_B_sep)

# Entangled state: |Φ+⟩
qc_ent = QuantumCircuit(2)
qc_ent.h(1)
qc_ent.cx(1, 0)
dm_ent = DensityMatrix(Statevector.from_instruction(qc_ent))
rho_A_ent = partial_trace(dm_ent, [0])
rho_B_ent = partial_trace(dm_ent, [1])
bloch_A_ent = dm_to_bloch(rho_A_ent)
bloch_B_ent = dm_to_bloch(rho_B_ent)

print(f"Separable |+0⟩:")
print(f"  Qubit 1 (|+⟩): Bloch = ({bloch_A_sep[0]:.2f}, {bloch_A_sep[1]:.2f}, {bloch_A_sep[2]:.2f})")
print(f"  Qubit 0 (|0⟩): Bloch = ({bloch_B_sep[0]:.2f}, {bloch_B_sep[1]:.2f}, {bloch_B_sep[2]:.2f})")
print()
print(f"Entangled |Φ+⟩:")
print(f"  Qubit 1: Bloch = ({bloch_A_ent[0]:.2f}, {bloch_A_ent[1]:.2f}, {bloch_A_ent[2]:.2f}) ← AT ORIGIN!")
print(f"  Qubit 0: Bloch = ({bloch_B_ent[0]:.2f}, {bloch_B_ent[1]:.2f}, {bloch_B_ent[2]:.2f}) ← AT ORIGIN!")
print()
print("💡 Entangled qubits have NO individual identity.")
print("   Their Bloch vectors collapse to the center = maximally mixed.")
print("   All information is in the CORRELATIONS between qubits.")

In [ ]:
# ── Side-by-side Bloch sphere visualization ──
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Separable |+0⟩ — Definite Bloch Vectors',
                    'Entangled |Φ+⟩ — Bloch Vectors at Origin')
)

# Sphere wireframe for both subplots
u = np.linspace(0, 2*np.pi, 30)
v = np.linspace(0, np.pi, 30)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones(len(u)), np.cos(v))

for col in [1, 2]:
    fig.add_trace(go.Surface(
        x=xs, y=ys, z=zs, opacity=0.08,
        colorscale=[[0, 'lightgray'], [1, 'lightgray']],
        showscale=False, hoverinfo='skip'
    ), row=1, col=col)
    # Axes
    for coords, color in [([1.3,0,0], 'red'), ([0,1.3,0], 'green'), ([0,0,1.3], 'blue')]:
        fig.add_trace(go.Scatter3d(
            x=[0, coords[0]], y=[0, coords[1]], z=[0, coords[2]],
            mode='lines', line=dict(color=color, width=2),
            showlegend=False, hoverinfo='skip'
        ), row=1, col=col)

# LEFT: Separable — Bloch vectors point somewhere definite
for bloch, color, name in [
    (bloch_A_sep, 'red', 'Qubit 1 (|+⟩)'),
    (bloch_B_sep, 'blue', 'Qubit 0 (|0⟩)')
]:
    # Arrow from origin to Bloch vector
    fig.add_trace(go.Scatter3d(
        x=[0, bloch[0]], y=[0, bloch[1]], z=[0, bloch[2]],
        mode='lines', line=dict(color=color, width=6),
        name=name, showlegend=True
    ), row=1, col=1)
    fig.add_trace(go.Scatter3d(
        x=[bloch[0]], y=[bloch[1]], z=[bloch[2]],
        mode='markers+text', text=[name],
        textposition='top center',
        marker=dict(size=8, color=color, symbol='diamond'),
        showlegend=False
    ), row=1, col=1)

# RIGHT: Entangled — Bloch vectors at origin (mixed state)
for bloch, color, name in [
    (bloch_A_ent, 'red', 'Qubit 1 (mixed)'),
    (bloch_B_ent, 'blue', 'Qubit 0 (mixed)')
]:
    fig.add_trace(go.Scatter3d(
        x=[bloch[0]], y=[bloch[1]], z=[bloch[2]],
        mode='markers+text', text=[name],
        textposition='top center',
        marker=dict(size=12, color=color, symbol='circle', opacity=0.7),
        name=name, showlegend=True
    ), row=1, col=2)

# Add "origin" marker for entangled
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers+text', text=['Both at origin!'],
    textposition='bottom center',
    marker=dict(size=6, color='black'),
    showlegend=False
), row=1, col=2)

scene_settings = dict(
    xaxis=dict(range=[-1.5, 1.5], title='X'),
    yaxis=dict(range=[-1.5, 1.5], title='Y'),
    zaxis=dict(range=[-1.5, 1.5], title='Z'),
    aspectmode='cube'
)
fig.update_layout(
    title='Separable vs Entangled: Individual Qubit Bloch Vectors',
    scene=scene_settings, scene2=scene_settings,
    width=1200, height=600
)
fig.show()

---
## Section 5: Partial Trace & Reduced Density Matrices

The **partial trace** answers: *"What does qubit A look like if I ignore qubit B?"*

For a 2-qubit density matrix $\rho_{AB}$:
$$\rho_A = \text{Tr}_B(\rho_{AB})$$

### Purity Test
$$\text{Purity} = \text{Tr}(\rho_A^2)$$

| Purity | Meaning |
| :--- | :--- |
| 1.0 | Pure state → separable (qubit has definite state) |
| 0.5 | Maximally mixed → maximally entangled |
| Between | Partially entangled |

In [ ]:
print("📊 Partial Trace & Purity Analysis")
print("=" * 55)

test_states = {
    '|00⟩ (separable)': (QuantumCircuit(2), ),
    '|+0⟩ (separable)': (QuantumCircuit(2), 'h1'),
    '|Φ+⟩ (max entangled)': (QuantumCircuit(2), 'h1', 'cx'),
}

# Build circuits
circuits = {}
qc = QuantumCircuit(2)
circuits['|00⟩ (separable)'] = qc

qc = QuantumCircuit(2); qc.h(1)
circuits['|+0⟩ (separable)'] = qc

qc = QuantumCircuit(2); qc.h(1); qc.cx(1, 0)
circuits['|Φ+⟩ (max entangled)'] = qc

# Partially entangled: Ry(π/4) + CNOT
qc = QuantumCircuit(2); qc.ry(np.pi/4, 1); qc.cx(1, 0)
circuits['Ry(π/4)+CNOT (partial)'] = qc

print(f"{'State':<30s} {'Purity':>8s}  {'Entangled?':>10s}  Reduced ρ_A")
print("-" * 75)

for name, qc in circuits.items():
    sv = Statevector.from_instruction(qc)
    dm = DensityMatrix(sv)
    reduced = partial_trace(dm, [0])  # trace out qubit 0
    purity = np.real(np.trace(reduced.data @ reduced.data))
    entangled = purity < 0.999
    rho_diag = f"diag=[{reduced.data[0,0].real:.3f}, {reduced.data[1,1].real:.3f}]"
    status = "YES ❌" if entangled else "NO  ✅"
    print(f"{name:<30s} {purity:8.4f}  {status:>10s}  {rho_diag}")

print()
print("💡 Purity = 1 → pure → separable (qubit has its own state)")
print("   Purity = 0.5 → maximally mixed → maximally entangled")
print("   The more entangled, the less you can say about each qubit alone.")

---
## Section 6: GHZ and W States — Two Types of Entanglement

For 3+ qubits, there are **fundamentally different kinds** of entanglement:

### GHZ State
$$\vert\text{GHZ}\rangle = \frac{\vert 000\rangle + \vert 111\rangle}{\sqrt{2}}$$
- "All or nothing" — lose one qubit and the rest are **completely separable**
- Like a house of cards: fragile

### W State  
$$\vert W\rangle = \frac{\vert 001\rangle + \vert 010\rangle + \vert 100\rangle}{\sqrt{3}}$$
- "Distributed" — lose one qubit and the rest are **still entangled**
- Like a woven fabric: robust

These are **not equivalent** — you cannot convert one to the other using only local operations (different SLOCC classes).

In [ ]:
print("🔺 GHZ vs W: Two Types of Multi-Qubit Entanglement")
print("=" * 55)

# ── GHZ state ──
qc_ghz = QuantumCircuit(3)
qc_ghz.h(2)
qc_ghz.cx(2, 1)
qc_ghz.cx(2, 0)

sv_ghz = Statevector.from_instruction(qc_ghz)
print("GHZ Circuit:")
print(qc_ghz.draw())
print(f"\nGHZ state: {np.array(sv_ghz).round(4)}")
print(f"  = (|000⟩ + |111⟩)/√2")

# What happens if we lose qubit 2? (trace it out)
dm_ghz = DensityMatrix(sv_ghz)
reduced_ghz_12 = partial_trace(dm_ghz, [2])  # trace out qubit 2
purity_ghz_remaining = np.real(np.trace(reduced_ghz_12.data @ reduced_ghz_12.data))

# Check if remaining 2 qubits are entangled
reduced_ghz_1 = partial_trace(dm_ghz, [0, 2])  # just qubit 1
purity_ghz_1 = np.real(np.trace(reduced_ghz_1.data @ reduced_ghz_1.data))
print(f"\n  Lose qubit 2 → remaining purity: {purity_ghz_remaining:.4f}")
print(f"  Remaining qubits 0,1 are {'SEPARABLE' if purity_ghz_1 > 0.99 else 'entangled'}")
print("  → GHZ is FRAGILE: lose one qubit → all entanglement gone")

print()

# ── W state ──
qc_w = QuantumCircuit(3)
target_w = np.zeros(8, dtype=complex)
target_w[1] = 1/np.sqrt(3)  # |001⟩
target_w[2] = 1/np.sqrt(3)  # |010⟩
target_w[4] = 1/np.sqrt(3)  # |100⟩
qc_w.initialize(target_w)

sv_w = Statevector.from_instruction(qc_w)
print("W state: (|001⟩ + |010⟩ + |100⟩)/√3")
print(f"  Amplitudes: {np.array(sv_w).round(4)}")

# What happens if we lose qubit 2?
dm_w = DensityMatrix(sv_w)
reduced_w_01 = partial_trace(dm_w, [2])  # trace out qubit 2

# Check purity of just one remaining qubit
reduced_w_1 = partial_trace(dm_w, [0, 2])  # just qubit 1
purity_w_1 = np.real(np.trace(reduced_w_1.data @ reduced_w_1.data))

# Check if remaining pair is entangled
reduced_w_0_from_pair = partial_trace(reduced_w_01, [1])
purity_pair = np.real(np.trace(reduced_w_0_from_pair.data @ reduced_w_0_from_pair.data))
print(f"\n  Lose qubit 2 → remaining qubit purity: {purity_pair:.4f}")
print(f"  Remaining qubits 0,1 are {'separable' if purity_pair > 0.99 else 'STILL ENTANGLED'}")
print("  → W is ROBUST: lose one qubit → others stay entangled")

print()
print("=" * 55)
print("Summary:")
print("  GHZ: All-or-nothing entanglement (fragile, useful for QEC)")
print("  W:   Distributed entanglement (robust, useful for quantum memories)")
print("  They're in DIFFERENT entanglement classes — fundamentally distinct!")

---
## Section 7: The No-Cloning Theorem

**Theorem:** There is no quantum operation that can copy an **arbitrary** unknown quantum state.

$$U\vert\psi\rangle\vert 0\rangle = \vert\psi\rangle\vert\psi\rangle \quad \leftarrow \text{IMPOSSIBLE for all } \vert\psi\rangle$$

### Why?
By **linearity**: if $U\vert 0\rangle\vert 0\rangle = \vert 00\rangle$ and $U\vert 1\rangle\vert 0\rangle = \vert 11\rangle$, then:

$$U\vert +\rangle\vert 0\rangle = \frac{\vert 00\rangle + \vert 11\rangle}{\sqrt{2}} = \vert\Phi^+\rangle \neq \vert +\rangle\vert +\rangle$$

CNOT creates **entanglement**, not a **copy**!

### Why This Is a Feature
No-cloning enables **quantum cryptography (QKD)**: an eavesdropper cannot copy quantum states without disturbing them.

In [ ]:
print("🚫 The No-Cloning Theorem")
print("=" * 55)

def attempt_clone(state_name):
    """Attempt to clone using CNOT."""
    qc = QuantumCircuit(2)
    qc_single = QuantumCircuit(1)

    if state_name == '1':
        qc.x(0); qc_single.x(0)
    elif state_name == '+':
        qc.h(0); qc_single.h(0)
    elif state_name == '-':
        qc.x(0); qc.h(0)
        qc_single.x(0); qc_single.h(0)

    qc.cx(0, 1)  # attempt to clone

    result = Statevector.from_instruction(qc)
    single = np.array(Statevector.from_instruction(qc_single))
    ideal = Statevector(np.kron(single, single))

    return result, ideal

print(f"{'State':<8s} {'CNOT result':<35s} {'Ideal |ψ⟩|ψ⟩':<35s} {'Cloned?'}")
print("-" * 90)

for state in ['0', '1', '+', '-']:
    result, ideal = attempt_clone(state)
    r_arr = np.array(result).round(4)
    i_arr = np.array(ideal).round(4)
    match = result.equiv(ideal)
    symbol = "✅ Yes" if match else "❌ No → Entangled!"
    print(f"|{state}⟩     {str(r_arr):<35s} {str(i_arr):<35s} {symbol}")

print()
print("💡 CNOT copies BASIS states (|0⟩, |1⟩) but ENTANGLES superpositions.")
print("   This is NOT a bug — it's what enables quantum cryptography!")
print("   An eavesdropper cannot copy quantum states without detection.")

---
## Section 8: Bell State Measurement Correlations

Bell states exhibit **perfect correlations** between measurement outcomes:

| Bell State | Correlation | Meaning |
| :--- | :--- | :--- |
| $\vert\Phi^+\rangle$ = ($\vert 00\rangle$ + $\vert 11\rangle$)/$\sqrt{2}$ | +1 | Always **same** outcome |
| $\vert\Phi^-\rangle$ = ($\vert 00\rangle$ - $\vert 11\rangle$)/$\sqrt{2}$ | +1 | Always **same** outcome |
| $\vert\Psi^+\rangle$ = ($\vert 01\rangle$ + $\vert 10\rangle$)/$\sqrt{2}$ | −1 | Always **opposite** outcome |
| $\vert\Psi^-\rangle$ = ($\vert 01\rangle$ - $\vert 10\rangle$)/$\sqrt{2}$ | −1 | Always **opposite** outcome |

These correlations are **stronger than anything classical** — this is the basis of **Bell's theorem** and the **CHSH inequality**.

In [ ]:
print("📈 Bell State Measurement Correlations")
print("=" * 55)

bell_configs = {
    '|Φ+⟩': make_bell('Φ+'),
    '|Φ-⟩': make_bell('Φ-', [1]),
    '|Ψ+⟩': make_bell('Ψ+', [0]),
    '|Ψ-⟩': make_bell('Ψ-', [0, 1]),
}

print(f"{'Bell State':<12s} {'P(00)':>7s} {'P(01)':>7s} {'P(10)':>7s} {'P(11)':>7s} {'Corr':>7s}")
print("-" * 55)

correlation_data = {}
for name, qc in bell_configs.items():
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    p00 = probs.get('00', 0)
    p01 = probs.get('01', 0)
    p10 = probs.get('10', 0)
    p11 = probs.get('11', 0)
    corr = (p00 + p11) - (p01 + p10)
    correlation_data[name] = corr
    print(f"{name:<12s} {p00:7.3f} {p01:7.3f} {p10:7.3f} {p11:7.3f} {corr:+7.3f}")

print()

# Simulate sampling to show correlations
print("Sampling |Φ+⟩ (20 measurements):")
sv_phi_plus = Statevector.from_instruction(bell_configs['|Φ+⟩'])
counts = sv_phi_plus.sample_counts(20)
outcomes = []
for outcome, count in sorted(counts.items()):
    outcomes.extend([outcome] * count)
print(f"  Outcomes: {' '.join(outcomes)}")
print(f"  → Always same! (00 or 11)")

print()
print("Sampling |Ψ-⟩ (20 measurements):")
sv_psi_minus = Statevector.from_instruction(bell_configs['|Ψ-⟩'])
counts = sv_psi_minus.sample_counts(20)
outcomes = []
for outcome, count in sorted(counts.items()):
    outcomes.extend([outcome] * count)
print(f"  Outcomes: {' '.join(outcomes)}")
print(f"  → Always opposite! (01 or 10)")

print()
print("💡 These correlations are INSTANTANEOUS and hold regardless of distance.")
print("   This is what Einstein called 'spooky action at a distance.'")
print("   Bell's theorem proves NO local hidden variable theory can explain this.")

In [ ]:
# ── Visualize correlations as bar chart ──
fig = go.Figure()

for name, qc in bell_configs.items():
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    outcomes = ['00', '01', '10', '11']
    p_vals = [probs.get(o, 0) for o in outcomes]
    fig.add_trace(go.Bar(
        x=outcomes, y=p_vals, name=name,
        text=[f'{p:.2f}' for p in p_vals], textposition='auto'
    ))

fig.update_layout(
    title='Bell State Measurement Probabilities',
    xaxis_title='Measurement Outcome',
    yaxis_title='Probability',
    barmode='group',
    width=800, height=500,
    legend=dict(font=dict(size=14))
)
fig.show()

---
## 🎯 Day 3 Summary & Deliverables

### What you learned today:
1. **Tensor Products** — Multi-qubit states live in $\mathbb{C}^{2^n}$ (exponential growth!)
2. **CNOT Gate** — The fundamental entangling gate: superposition + CNOT = entanglement
3. **Bell States** — The 4 maximally entangled 2-qubit states, built with H + CNOT
4. **Entanglement Visualization** — Entangled qubits have no individual Bloch vectors
5. **Partial Trace** — Purity test: Tr(ρ²) = 1 → separable, 0.5 → maximally entangled
6. **GHZ vs W** — Two fundamentally different types of multi-qubit entanglement
7. **No-Cloning** — CNOT copies basis states but entangles superpositions
8. **Bell Correlations** — Perfect (anti-)correlations, stronger than any classical system

### Your deliverables:
- ✅ All 4 Bell state circuits with verification
- ✅ Entanglement detection via partial trace
- ✅ Bloch sphere visualization (separable vs entangled)
- ✅ GHZ and W state comparison
- ✅ No-cloning theorem demonstration
- ✅ Bell measurement correlation analysis
- ✅ Portfolio artifact: "Entanglement: The Resource That Makes Quantum Computing Possible"

### Key takeaway for interviews:
> "Entanglement is what separates quantum from classical. Without it, quantum circuits can be efficiently simulated classically (Gottesman-Knill). With entanglement plus non-Clifford gates, we access an exponentially large state space that classical computers cannot efficiently represent. The CNOT gate is the workhorse — but it's also the most expensive gate on hardware, making CNOT count the key optimization metric."

---
**Tomorrow (Day 4):** Quantum Circuits as Matrix Operations — multi-qubit gate matrices, controlled gates, universal gate sets, and the Qiskit Aer simulator.